# LDA Topic Analysis

In [ ]:
import pandas as pd
import ast

df = pd.read_csv('datasets/labeled_vulnerabilities.csv')
df['description_cleaned'] = df['description_cleaned'].apply(ast.literal_eval)
df.head()


C:\Users\kevin\AppData\Local\Temp\ipykernel_25852\2773267133.py:4: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('datasets/labeled_vulnerabilities.csv')


,id,description,datePublished,dateUpdated,baseScoreVersion,exploitedSince,baseScoreVector,epss,assigner,aliases,enisaIdVendor,references,enisaIdProduct,baseScore,enisaUuid,description_cleaned,product_name,vendor_name,product_label
0,EUVD-2026-24141,FreeScout is a free self-hosted help desk and ...,"Apr 21, 2026, 3:52:39 PM","Apr 21, 2026, 3:52:39 PM",3.1,NaN,CVSS:3.1/AV:N/AC:L/PR:N/UI:R/S:C/C:L/I:L/A:N,0.0,GitHub_M,CVE-2026-40565\n,[{'id': 'df8ea7fe-ea1b-3f31-a494-89b12dbf0c16'...,https://github.com/freescout-help-desk/freesco...,[{'id': '35cc8fe9-cc82-3bc7-9dda-d8dfd8a9d826'...,6.1,b3a6ee31-e20e-3b1c-a684-768095acc098,"[freescout, free, selfhosted, help, desk, shar...",freescout,freescout-help-desk,Customer Support & Help Desk Software
1,EUVD-2026-24138,Vulnerability related to an unquoted search pa...,"Apr 21, 2026, 3:32:22 PM","Apr 21, 2026, 3:32:22 PM",4.0,NaN,CVSS:4.0/AV:L/AC:L/AT:N/PR:L/UI:N/VC:H/VI:H/VA...,0.0,INCIBE,GHSA-9vxj-j2f7-9mgg\nCVE-2026-5789\n,[{'id': '5112208e-9449-3f30-b5a0-bdefa26b6740'...,https://www.incibe.es/en/incibe-cert/notices/a...,[{'id': '4dd00402-1cb8-33d1-91f8-5daf0e5fa5c5'...,8.5,7ed70757-c092-3a0b-99e2-96a7f815201b,"[vulnerability, related, unquoted, search, pat...",civetweb,CivetWeb,Web Server Software
2,EUVD-2026-24136,"The method ""sock_recvfrom_into()"" of ""asyncio....","Apr 21, 2026, 3:32:22 PM","Apr 21, 2026, 3:32:22 PM",4.0,NaN,CVSS:4.0/AV:N/AC:L/AT:N/PR:N/UI:N/VC:L/VI:L/VA...,0.0,PSF,GHSA-3p9c-22jr-wq4x\nCVE-2026-3298\n,[{'id': '455eddad-0274-38e3-876c-1da485aedea5'...,https://github.com/python/cpython/pull/148809\...,[{'id': '79323943-f5b6-3a59-ac3e-e7849e5be63e'...,8.8,e0f49fce-f7e8-3dfc-b70e-dbc0c058a383,"[method, sock_recvfrom_into, asyncioproacterev...",CPython,Python Software Foundation,Programming Language Runtimes & Interpreters
3,EUVD-2026-24130,User‑Controlled HTTP Header in Fortra's GoAnyw...,"Apr 21, 2026, 3:32:22 PM","Apr 21, 2026, 3:32:22 PM",3.1,NaN,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:L/I:N/A:L,0.0,Fortra,GHSA-6x5f-r479-qh4p\nCVE-2026-1089\n,[{'id': '2ae1e1ad-e03a-3855-b5d6-a2176c0b31dd'...,https://www.fortra.com/security/advisories/pro...,[{'id': '70404ca1-7e53-3be1-87b1-6dc09b1e4ff2'...,6.5,f1bfb170-4572-33bb-a02c-83c7a5e4ce08,"[usercontrolled, http, header, fortras, goanyw...",GoAnywhere MFT,Fortra,FTP & Managed File Transfer Software
4,EUVD-2026-24129,The login limit is not enforced on the SFTP se...,"Apr 21, 2026, 3:32:22 PM","Apr 21, 2026, 3:32:22 PM",3.1,NaN,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:L/I:L/A:L,0.0,Fortra,GHSA-rpc6-m3h5-gmf2\nCVE-2026-0972\n,[{'id': 'cba00c9f-fa23-3062-a6a5-5709e85421d7'...,https://fortra.com/security/advisories/product...,[{'id': 'd104a637-0939-344c-be4c-448beede3920'...,7.3,3965ce35-f611-3fd6-bc4c-580538b77705,"[login, limit, enforced, sftp, service, fortra...",GoAnywhere MFT,Fortra,FTP & Managed File Transfer Software


## Vectorization

In [4]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary
import numpy as np
import itertools

tokenized_docs = df["description_cleaned"].tolist()
texts = [" ".join(doc) for doc in tokenized_docs]

count_vectorizer = CountVectorizer(
    max_df=0.5,
    min_df=10,
    max_features=50000,
    ngram_range=(1, 2),
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z0-9_\-]{2,}\b",
    stop_words='english',
)
dtm_counts = count_vectorizer.fit_transform(texts)
count_features = count_vectorizer.get_feature_names_out()

print(f"Vocabulary size: {len(count_features)}")
print(f"Document-term matrix shape: {dtm_counts.shape}")

gensim_dict = Dictionary(tokenized_docs)


Vocabulary size: 42604
Document-term matrix shape: (130193, 42604)


In [ ]:
def get_top_words_per_topic(model, feature_names, n_words=10):
    topics = []
    for topic in model.components_:
        top_indices = topic.argsort()[:-n_words - 1:-1]
        topics.append([feature_names[i] for i in top_indices])
    return topics


def topic_diversity(topics):
    all_words = [w for topic in topics for w in topic]
    if not all_words:
        return 0.0
    return len(set(all_words)) / len(all_words)


def coherence_score(topics, tokenized_docs, dictionary):
    cm = CoherenceModel(
        topics=topics,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence='c_v',
    )
    return cm.get_coherence()


## Grid search hyperparameters

In [ ]:
k_range = [10, 15, 20, 25, 30, 35, 40]
alpha_range = [0.05, 0.1, 0.3]
beta_range = [None, 0.01, 0.1]  # None = sklearn auto

lda_results = []

for k, alpha, beta in itertools.product(k_range, alpha_range, beta_range):
    lda = LatentDirichletAllocation(
        n_components=k,
        doc_topic_prior=alpha,
        topic_word_prior=beta,
        max_iter=20,
        learning_method='batch',
        random_state=1,
    )
    lda.fit(dtm_counts)

    topics = get_top_words_per_topic(lda, count_features, n_words=10)
    coherence = coherence_score(topics, tokenized_docs, gensim_dict)
    diversity = topic_diversity(topics)

    lda_results.append({
        'k': k,
        'alpha': alpha,
        'beta': beta,
        'coherence': coherence,
        'diversity': diversity,
        'combined_score': coherence * diversity, #to get a score between 0 and 1, with both contributing equally
    })
    print(f"k={k:>2} alpha={alpha} beta={beta}: "
          f"coherence={coherence:.4f} diversity={diversity:.3f}")

lda_results_df = pd.DataFrame(lda_results).sort_values('combined_score', ascending=False)



k=10 alpha=0.05 beta=None: coherence=0.7507 diversity=0.680
k=10 alpha=0.05 beta=0.01: coherence=0.7507 diversity=0.680
k=10 alpha=0.05 beta=0.1: coherence=0.7507 diversity=0.680
k=10 alpha=0.1 beta=None: coherence=0.7507 diversity=0.680
k=10 alpha=0.1 beta=0.01: coherence=0.7507 diversity=0.680
k=10 alpha=0.1 beta=0.1: coherence=0.7507 diversity=0.680
k=10 alpha=0.3 beta=None: coherence=0.7288 diversity=0.690
k=10 alpha=0.3 beta=0.01: coherence=0.7245 diversity=0.700
k=10 alpha=0.3 beta=0.1: coherence=0.7288 diversity=0.690
k=15 alpha=0.05 beta=None: coherence=0.7168 diversity=0.640
k=15 alpha=0.05 beta=0.01: coherence=0.7199 diversity=0.660
k=15 alpha=0.05 beta=0.1: coherence=0.7152 diversity=0.633
k=15 alpha=0.1 beta=None: coherence=0.7234 diversity=0.667
k=15 alpha=0.1 beta=0.01: coherence=0.7234 diversity=0.667
k=15 alpha=0.1 beta=0.1: coherence=0.7205 diversity=0.667
k=15 alpha=0.3 beta=None: coherence=0.7142 diversity=0.687
k=15 alpha=0.3 beta=0.01: coherence=0.7142 diversity=0.

## Showing best configuration

In [5]:
print("Top 10 configs (by coherence * diversity):")
print(lda_results_df.head(10).to_string(index=False))

best_config = lda_results_df.iloc[0]
print(f"\nBest config: k={int(best_config['k'])}, alpha={best_config['alpha']}, "
      f"beta={best_config['beta']} "
      f"-> coherence={best_config['coherence']:.4f}, diversity={best_config['diversity']:.3f}")


Top 10 configs (by coherence * diversity):
 k  alpha  beta  coherence  diversity  combined_score
30   0.30   NaN   0.731982   0.706667        0.517267
30   0.30  0.01   0.730677   0.700000        0.511474
10   0.05   NaN   0.750720   0.680000        0.510490
10   0.10   NaN   0.750720   0.680000        0.510490
10   0.10  0.01   0.750720   0.680000        0.510490
10   0.05  0.01   0.750720   0.680000        0.510490
10   0.05  0.10   0.750720   0.680000        0.510490
10   0.10  0.10   0.750720   0.680000        0.510490
30   0.30  0.10   0.732252   0.693333        0.507694
10   0.30  0.01   0.724505   0.700000        0.507153

Best config: k=30, alpha=0.3, beta=nan -> coherence=0.7320, diversity=0.707


In [ ]:
def print_topics(model, feature_names, n_words=10):
    for i, topic in enumerate(model.components_):
        top_words = [feature_names[j] for j in topic.argsort()[:-n_words - 1:-1]]
        print(f"Topic {i:02d}: {', '.join(top_words)}")

lda_30_None = LatentDirichletAllocation(
    n_components=30,
    doc_topic_prior=0.3,
    topic_word_prior=None,
    max_iter=20,
    learning_method='batch',
    random_state=1,
)
lda_30_None.fit(dtm_counts)

print("=== k=30, alpha=0.3, beta=None ===")
print_topics(lda_30_None, count_features, n_words=10)

=== k=30, alpha=0.3, beta=None ===
Topic 00: kernel, path, memory, linux, following, traversal, fix, resolved, linux kernel, path traversal
Topic 01: xss, scripting, crosssite scripting, crosssite, malicious, scripting xss, xss vulnerability, script, stored, javascript
Topic 02: oracle, access, cvss, attack, impact, data, successful, successful attack, unauthorized, product
Topic 03: ibm, session, xforce, ibm xforce, vulnerable, user, potentially leading, leading, potentially, allows
Topic 04: earlier, user, version, file, version earlier, requires, exploitation, open, interaction, affected
Topic 05: information, access, sensitive, attacker, sensitive information, data, disclosure, information disclosure, device, file
Topic 06: crafted, specially, specially crafted, ibm, attacker, exists, vulnerability exists, request, trigger, http
Topic 07: issue, macos, app, fixed, issue fixed, addressed, able, issue addressed, improved, addressed improved
Topic 08: sql, issue, affect, issue affect,

In [ ]:
lda_30_001 = LatentDirichletAllocation(
    n_components=30,
    doc_topic_prior=0.3,
    topic_word_prior=0.01,
    max_iter=20,
    learning_method='batch',
    random_state=1,
)
lda_30_001.fit(dtm_counts)

print("=== k=30, alpha=0.3, beta=0.01 ===")
print_topics(lda_30_001, count_features, n_words=10)

=== k=30, alpha=0.3, beta=0.01 ===
Topic 00: path, kernel, memory, linux, following, traversal, fix, resolved, linux kernel, path traversal
Topic 01: xss, scripting, crosssite scripting, crosssite, malicious, scripting xss, xss vulnerability, script, stored, javascript
Topic 02: oracle, access, cvss, attack, impact, data, successful, successful attack, unauthorized, base
Topic 03: ibm, session, xforce, ibm xforce, vulnerable, user, potentially leading, potentially, leading, allows
Topic 04: earlier, user, version, file, version earlier, requires, exploitation, open, interaction, affected
Topic 05: information, access, sensitive, attacker, sensitive information, data, disclosure, information disclosure, device, file
Topic 06: crafted, specially, specially crafted, attacker, ibm, exists, vulnerability exists, request, trigger, http
Topic 07: issue, macos, app, fixed, issue fixed, addressed, able, issue addressed, improved, addressed improved
Topic 08: issue, sql, affect, issue affect, al

In [ ]:

lda_10 = LatentDirichletAllocation(
    n_components=10,
    doc_topic_prior=0.05,
    topic_word_prior=None,
    max_iter=20,
    learning_method='batch',
    random_state=1,
)
lda_10.fit(dtm_counts)

print("=== k=10, alpha=0.05, beta=None ===")
print_topics(lda_10, count_features, n_words=10)

=== k=10, alpha=0.05, beta=None ===
Topic 00: file, privilege, path, lead, memory, user, service, needed, execution, local
Topic 01: xss, version, scripting, crosssite, attacker, crosssite scripting, malicious, scripting xss, user, javascript
Topic 02: oracle, access, cvss, impact, attack, successful, data, server, attacker, unauthorized
Topic 03: attack, manipulation, file, exploit, remotely, lead, used, argument, public, manipulation argument
Topic 04: version, user, command, privilege, earlier, attacker, arbitrary, file, allow, malicious
Topic 05: attacker, version, access, allow, prior, device, exploit, version prior, affected, allows
Topic 06: code, attacker, execute, remote, arbitrary, file, execution, code execution, ibm, arbitrary code
Topic 07: issue, issue affect, affect, allows, crosssite, improper, page, neutralization, improper neutralization, web
Topic 08: version, user, issue, file, server, prior, allows, attacker, request, sql
Topic 09: plugin, wordpress, version, inclu

## Fitting the model


In [5]:
final_model = LatentDirichletAllocation(
    n_components=30,
    doc_topic_prior=0.3,
    topic_word_prior=None,
    max_iter=20,
    learning_method='batch',
    random_state=1,
)
final_model.fit(dtm_counts)


,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",30
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",0.3
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'batch'
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",20
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",128
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",-1
,"total_samples total_samples: int, default=1e6Total number of documents. Only used in the :meth:`partial_fit` method.",1000000.0
,"perp_tol perp_tol: float, default=1e-1Perplexity tolerance. Only used when ``evaluate_every`` is greater than 0.",0.1


## Analysis

In [7]:
doc_topic_dist = final_model.transform(dtm_counts)  

In [ ]:
import pandas as pd
from scipy.spatial.distance import jensenshannon

doc_topic_df = pd.DataFrame(doc_topic_dist)
doc_topic_df['category'] = df['product_label'] 

category_profiles = doc_topic_df.groupby('category').mean() 

n = len(category_profiles)
dist_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        dist_matrix[i, j] = jensenshannon(category_profiles.iloc[i], category_profiles.iloc[j])

dist_df = pd.DataFrame(dist_matrix, index=category_profiles.index, columns=category_profiles.index)

In [9]:
dist_df

category,"AI Writing, Content & Code Assistant Tools",API & Integration Tools,Accounting & Financial Management Software,Application Server / Middleware,Artificial Intelligence & Machine Learning Software,Automotive & Vehicle Systems,Blockchain & Cryptocurrency Software,Booking & Reservation Systems (Non-WordPress),Building Management & Automation Systems,CI/CD & DevOps Automation Tools,...,Wearable & Consumer Electronics,Web Applications (General/Unclassified),Web Browsers & Extensions,Web Development Frameworks & Frontend Build Tools,Web Forums & Discussion Platforms,Web Scraping & Data Extraction Tools,Web Server Software,Website Builders & Templates,Wiki & Knowledge Base Software,WordPress / WooCommerce Plugins & Themes
category,,,,,,,,,,,,,,,,,,,,,
"AI Writing, Content & Code Assistant Tools",0.000000,0.215801,0.361821,0.362281,0.221055,0.375401,0.279571,0.472327,0.239865,0.199097,...,0.330611,0.307510,0.443372,0.189376,0.302321,0.174551,0.245879,0.369606,0.291352,0.421702
API & Integration Tools,0.215801,0.000000,0.328931,0.223669,0.239663,0.358750,0.243801,0.510186,0.238778,0.176502,...,0.256895,0.337939,0.428778,0.194856,0.278698,0.228361,0.183012,0.423054,0.258566,0.481372
Accounting & Financial Management Software,0.361821,0.328931,0.000000,0.317753,0.343645,0.418733,0.420215,0.337643,0.304847,0.394791,...,0.413751,0.251508,0.508249,0.315986,0.348998,0.360189,0.303854,0.282011,0.351643,0.462643
Application Server / Middleware,0.362281,0.223669,0.317753,0.000000,0.365885,0.417813,0.365826,0.562967,0.332999,0.333624,...,0.341694,0.434663,0.481135,0.345619,0.393592,0.382149,0.280351,0.496765,0.361934,0.529126
Artificial Intelligence & Machine Learning Software,0.221055,0.239663,0.343645,0.365885,0.000000,0.361784,0.268696,0.473736,0.238772,0.253357,...,0.341191,0.311889,0.437868,0.229356,0.303321,0.235976,0.235028,0.363980,0.315196,0.497470
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Web Scraping & Data Extraction Tools,0.174551,0.228361,0.360189,0.382149,0.235976,0.387441,0.200536,0.443828,0.276440,0.181396,...,0.365313,0.267098,0.428577,0.095896,0.224608,0.000000,0.208890,0.366940,0.282751,0.469553
Web Server Software,0.245879,0.183012,0.303854,0.280351,0.235028,0.346483,0.192986,0.466127,0.257566,0.241407,...,0.272782,0.299570,0.447441,0.173914,0.216000,0.208890,0.000000,0.373536,0.278388,0.492397
Website Builders & Templates,0.369606,0.423054,0.282011,0.496765,0.363980,0.434452,0.443141,0.210819,0.349513,0.443199,...,0.448040,0.195465,0.524336,0.349964,0.365985,0.366940,0.373536,0.000000,0.395708,0.398146


In [10]:
# Get upper triangle indices (excludes diagonal/self and duplicate mirror pairs)
n = len(dist_df)
iu = np.triu_indices(n, k=1)  # k=1 skips the diagonal

pairs = pd.DataFrame({
    'category_a': dist_df.index[iu[0]],
    'category_b': dist_df.columns[iu[1]],
    'distance': dist_df.values[iu]
})

# Most distinct pairs (highest distance)
most_distinct = pairs.sort_values('distance', ascending=False).head(20)
print("=== Most distinct category pairs ===")
print(most_distinct.to_string(index=False))

# Most similar pairs (lowest distance, excluding identical category accidentally appearing twice if any)
most_similar = pairs.sort_values('distance', ascending=True).head(20)
print("\n=== Most similar category pairs ===")
print(most_similar.to_string(index=False))

=== Most distinct category pairs ===
                                            category_a                                             category_b  distance
                 PDF & Document Viewer/Editor Software        Restaurant, Hospitality & Food Service Software  0.666850
                 PDF & Document Viewer/Editor Software             Real Estate & Property Management Software  0.664266
         Booking & Reservation Systems (Non-WordPress)                  PDF & Document Viewer/Editor Software  0.653358
                 PDF & Document Viewer/Editor Software       Point of Sale (POS) & Retail Management Software  0.645895
                 PDF & Document Viewer/Editor Software               WordPress / WooCommerce Plugins & Themes  0.645516
          Inventory & Supply Chain Management Software                  PDF & Document Viewer/Editor Software  0.643947
                 Event Management & Ticketing Software                  PDF & Document Viewer/Editor Software  0.643337
   

In [12]:
# Mean distance to all other categories (excludes self since diagonal is 0)
distance_means = dist_df.mean(axis=1).sort_values(ascending=False)

top_20_distinct = distance_means.head(20).reset_index()
top_20_distinct.columns = ['category', 'mean_distance']

print(top_20_distinct.to_string(index=False))

                                              category  mean_distance
                 PDF & Document Viewer/Editor Software       0.572669
                 Image, Photo & Video Editing Software       0.521650
              WordPress / WooCommerce Plugins & Themes       0.483138
                             Web Browsers & Extensions       0.480254
Manufacturing Execution Systems & Engineering Software       0.476582
       Restaurant, Hospitality & Food Service Software       0.459883
            Real Estate & Property Management Software       0.457830
         Booking & Reservation Systems (Non-WordPress)       0.445737
                 Firmware (General/Unspecified Device)       0.434515
      Point of Sale (POS) & Retail Management Software       0.425951
          Inventory & Supply Chain Management Software       0.422908
          Programming Language Runtimes & Interpreters       0.420999
           Social Media Management & Integration Tools       0.415586
                    